# Project VI - Priority-A Full Astrometric Covariance Retrieval

This notebook retrieves and audits Gaia DR3 astrometric uncertainty/correlation inputs for exactly two Project VI `validation_priority_A` candidates. It builds covariance-ready inputs for later correlated astrometric Monte Carlo work, but does not run Monte Carlo sampling.

In [1]:
from pathlib import Path
import warnings

import io
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORT = ROOT / "report"
DATA_RAW.mkdir(exist_ok=True)
DATA_PROCESSED.mkdir(exist_ok=True)
REPORT.mkdir(exist_ok=True)

TARGET_IDS = ["3089847099636770560", "3089534353001157632"]
TARGET_ID_SET = set(TARGET_IDS)

RAW_OUT = DATA_RAW / "project_vi_priority_a_gaia_dr3_astrometry.csv"
PROCESSED_OUT = DATA_PROCESSED / "project_vi_priority_a_covariance_inputs.csv"
FIELD_INVENTORY_OUT = DATA_PROCESSED / "project_vi_priority_a_field_inventory.csv"
REPORT_OUT = REPORT / "project_vi_priority_a_covariance_retrieval.md"

GAIA_FIELDS = [
    "source_id", "ra", "ra_error", "dec", "dec_error", "parallax", "parallax_error",
    "pmra", "pmra_error", "pmdec", "pmdec_error",
    "ra_dec_corr", "ra_parallax_corr", "ra_pmra_corr", "ra_pmdec_corr",
    "dec_parallax_corr", "dec_pmra_corr", "dec_pmdec_corr",
    "parallax_pmra_corr", "parallax_pmdec_corr", "pmra_pmdec_corr",
    "radial_velocity", "radial_velocity_error", "ruwe", "astrometric_params_solved",
    "duplicated_source", "visibility_periods_used",
]

LAMOST_FIELDS = ["rv", "rv_err", "radial_velocity_error", "snrg", "snru", "snrr", "snri", "snrz", "quality", "flag"]

print("Project root:", ROOT)
print("Targets:", TARGET_IDS)

Project root: /Users/liors/Documents/research/gaia-lamost-galactic-archaeology
Targets: ['3089847099636770560', '3089534353001157632']


In [2]:
def source_key(series):
    return pd.Series(series).astype("string").str.replace(r"\.0$", "", regex=True)

FIELD_UNITS = {
    "source_id": "dimensionless",
    "ra": "deg", "ra_error": "mas", "dec": "deg", "dec_error": "mas",
    "parallax": "mas", "parallax_error": "mas",
    "pmra": "mas/yr", "pmra_error": "mas/yr", "pmdec": "mas/yr", "pmdec_error": "mas/yr",
    "radial_velocity": "km/s", "radial_velocity_error": "km/s", "rv": "km/s", "rv_err": "km/s",
    "ruwe": "dimensionless", "astrometric_params_solved": "dimensionless",
    "duplicated_source": "boolean", "visibility_periods_used": "count",
}
for corr in [f for f in GAIA_FIELDS if f.endswith("_corr")]:
    FIELD_UNITS[corr] = "correlation coefficient"

def intended_role(field):
    if field in ["parallax", "pmra", "pmdec"]:
        return "central value for later correlated astrometric MC"
    if field in ["parallax_error", "pmra_error", "pmdec_error"]:
        return "standard uncertainty for later parallax/pm covariance matrix"
    if field in ["parallax_pmra_corr", "parallax_pmdec_corr", "pmra_pmdec_corr"]:
        return "correlation coefficient for 3D parallax/pm covariance matrix"
    if field.endswith("_corr"):
        return "retrieved for full astrometric audit; not used in first 3D covariance matrix"
    if field in ["ra", "dec"]:
        return "coordinate central value for orbit pipeline"
    if field in ["ra_error", "dec_error"]:
        return "coordinate uncertainty audit; not used until unit/frame treatment is defined"
    if field in ["radial_velocity", "rv"]:
        return "radial-velocity central value audit"
    if field in ["radial_velocity_error", "rv_err"]:
        return "radial-velocity measured uncertainty audit"
    if field in ["ruwe", "astrometric_params_solved", "duplicated_source", "visibility_periods_used"]:
        return "quality/completeness audit"
    return "field availability audit"

local_records=[]
for path in sorted((ROOT / "data").rglob("*.csv")):
    if path.name.startswith("project_vi_priority_a_"):
        continue
    try:
        if path.name.startswith("lamost"):
            # LAMOST raw files may be pipe-separated. Read only a small chunk for schema and target matching where possible.
            sep = "|" if "lamost" in path.name else ","
            df = pd.read_csv(path, sep=sep, dtype="string", nrows=5000)
        else:
            df = pd.read_csv(path, dtype="string")
    except Exception:
        continue
    cols = list(df.columns)
    id_cols = [c for c in cols if c.lower() in ["source_id", "gaia_source_id", "combined_gaia_source_id"] or "source_id" in c.lower()]
    if not id_cols:
        # Some raw LAMOST larger files have no Gaia source id and cannot be target matched.
        for field in LAMOST_FIELDS:
            hits = [c for c in cols if field.lower() == c.lower() or field.lower() in c.lower()]
            for hit in hits:
                local_records.append({
                    "source_id": "target_match_unavailable", "field": hit, "source_table": str(path.relative_to(ROOT)),
                    "availability": "schema_only_no_target_source_id", "value": "", "unit": FIELD_UNITS.get(hit, "catalog-defined"),
                    "provenance": "local file schema", "missingness": "target rows cannot be matched by source_id",
                    "intended_mc_role": intended_role(hit),
                })
        continue
    matched_any=False
    for id_col in id_cols:
        keyed = source_key(df[id_col])
        sub = df[keyed.isin(TARGET_ID_SET)].copy()
        if sub.empty:
            continue
        matched_any=True
        sub["source_id_key"] = source_key(sub[id_col])
        fields_to_check = sorted(set(GAIA_FIELDS + LAMOST_FIELDS + [c for c in cols if any(token in c.lower() for token in ["rv", "snr", "quality", "flag"])]))
        for _, row in sub.iterrows():
            for field in fields_to_check:
                if field in cols:
                    value = row.get(field, pd.NA)
                    availability = "available" if pd.notna(value) and str(value) != "" else "column_present_value_missing"
                    missingness = "not missing" if availability == "available" else "value missing"
                else:
                    value = ""
                    availability = "missing_column"
                    missingness = "column missing"
                local_records.append({
                    "source_id": row["source_id_key"], "field": field, "source_table": str(path.relative_to(ROOT)),
                    "availability": availability, "value": "" if pd.isna(value) else str(value),
                    "unit": FIELD_UNITS.get(field, "catalog-defined"), "provenance": "local repository file",
                    "missingness": missingness, "intended_mc_role": intended_role(field),
                })
    if not matched_any:
        continue

local_audit = pd.DataFrame(local_records)
print("Local audit records:", local_audit.shape)
local_audit.head()

Local audit records: (28548, 9)


,source_id,field,source_table,availability,value,unit,provenance,missingness,intended_mc_role
0,3089534353001157632,astrometric_params_solved,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,dimensionless,local repository file,column missing,quality/completeness audit
1,3089534353001157632,dec,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,deg,local repository file,column missing,coordinate central value for orbit pipeline
2,3089534353001157632,dec_error,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,mas,local repository file,column missing,coordinate uncertainty audit; not used until u...
3,3089534353001157632,dec_parallax_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,retrieved for full astrometric audit; not used...
4,3089534353001157632,dec_pmdec_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,retrieved for full astrometric audit; not used...


In [3]:
# Compact local availability view for the exact Gaia DR3 fields requested in this stage.
requested_local = local_audit[local_audit["field"].isin(GAIA_FIELDS + ["rv", "rv_err", "radial_velocity_error", "snrg", "snru", "snrr", "snri", "snrz"])]
local_best = (
    requested_local.assign(rank=requested_local["availability"].map({"available": 0, "column_present_value_missing": 1, "missing_column": 2, "schema_only_no_target_source_id": 3}).fillna(4))
    .sort_values(["source_id", "field", "rank", "source_table"])
    .groupby(["source_id", "field"], as_index=False)
    .first()
    .drop(columns="rank")
)
local_best.head(30)

,source_id,field,source_table,availability,value,unit,provenance,missingness,intended_mc_role
0,3089534353001157632,astrometric_params_solved,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,dimensionless,local repository file,column missing,quality/completeness audit
1,3089534353001157632,dec,data/processed/project_ii_angular_momentum_can...,column_present_value_missing,,deg,local repository file,value missing,coordinate central value for orbit pipeline
2,3089534353001157632,dec_error,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,mas,local repository file,column missing,coordinate uncertainty audit; not used until u...
3,3089534353001157632,dec_parallax_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,retrieved for full astrometric audit; not used...
4,3089534353001157632,dec_pmdec_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,retrieved for full astrometric audit; not used...
5,3089534353001157632,dec_pmra_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,retrieved for full astrometric audit; not used...
6,3089534353001157632,duplicated_source,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,boolean,local repository file,column missing,quality/completeness audit
7,3089534353001157632,parallax,data/processed/gaia_lamost_candidate_diagnosti...,available,0.4587656346030645,mas,local repository file,not missing,central value for later correlated astrometric MC
8,3089534353001157632,parallax_error,data/processed/project_vi_mc_prototype_candida...,available,0.018229530984877475,mas,local repository file,not missing,standard uncertainty for later parallax/pm cov...
9,3089534353001157632,parallax_pmdec_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,correlation coefficient for 3D parallax/pm cov...


In [4]:
query = f"""
SELECT
    source_id,
    ra,
    ra_error,
    dec,
    dec_error,
    parallax,
    parallax_error,
    pmra,
    pmra_error,
    pmdec,
    pmdec_error,
    ra_dec_corr,
    ra_parallax_corr,
    ra_pmra_corr,
    ra_pmdec_corr,
    dec_parallax_corr,
    dec_pmra_corr,
    dec_pmdec_corr,
    parallax_pmra_corr,
    parallax_pmdec_corr,
    pmra_pmdec_corr,
    radial_velocity,
    radial_velocity_error,
    ruwe,
    astrometric_params_solved,
    duplicated_source,
    visibility_periods_used
FROM gaiadr3.gaia_source
WHERE source_id IN ({", ".join(TARGET_IDS)})
"""
print(query)
tap_url = "https://gea.esac.esa.int/tap-server/tap/sync"
payload = urllib.parse.urlencode({
    "REQUEST": "doQuery",
    "LANG": "ADQL",
    "FORMAT": "csv",
    "QUERY": query,
}).encode("utf-8")
request = urllib.request.Request(tap_url, data=payload, headers={"Content-Type": "application/x-www-form-urlencoded"})
with urllib.request.urlopen(request, timeout=120) as response:
    csv_text = response.read().decode("utf-8")
gaia = pd.read_csv(io.StringIO(csv_text), dtype={"source_id": "string"})
gaia["source_id"] = source_key(gaia["source_id"])
gaia = gaia.sort_values("source_id").reset_index(drop=True)
gaia.to_csv(RAW_OUT, index=False)
print("Saved", RAW_OUT, gaia.shape)
gaia


SELECT
    source_id,
    ra,
    ra_error,
    dec,
    dec_error,
    parallax,
    parallax_error,
    pmra,
    pmra_error,
    pmdec,
    pmdec_error,
    ra_dec_corr,
    ra_parallax_corr,
    ra_pmra_corr,
    ra_pmdec_corr,
    dec_parallax_corr,
    dec_pmra_corr,
    dec_pmdec_corr,
    parallax_pmra_corr,
    parallax_pmdec_corr,
    pmra_pmdec_corr,
    radial_velocity,
    radial_velocity_error,
    ruwe,
    astrometric_params_solved,
    duplicated_source,
    visibility_periods_used
FROM gaiadr3.gaia_source
WHERE source_id IN (3089847099636770560, 3089534353001157632)



Saved /Users/liors/Documents/research/gaia-lamost-galactic-archaeology/data/raw/project_vi_priority_a_gaia_dr3_astrometry.csv (2, 27)


,source_id,ra,ra_error,dec,dec_error,parallax,parallax_error,pmra,pmra_error,pmdec,...,dec_pmdec_corr,parallax_pmra_corr,parallax_pmdec_corr,pmra_pmdec_corr,radial_velocity,radial_velocity_error,ruwe,astrometric_params_solved,duplicated_source,visibility_periods_used
0,3089534353001157632,124.176994,0.015250,0.737449,0.011500,0.458766,0.018230,2.906364,0.018547,-36.283028,...,0.010915,0.231354,-0.248190,-0.106592,-48.846275,1.285616,1.197178,31,False,18
1,3089847099636770560,122.508355,0.020588,1.280094,0.017173,0.637151,0.023816,-42.072760,0.025911,-36.356421,...,-0.121074,0.173574,-0.119495,-0.219415,NaN,NaN,0.933814,31,False,18


In [5]:
assert len(gaia) == 2, f"Expected exactly 2 Gaia rows, got {len(gaia)}"
assert set(gaia["source_id"]) == TARGET_ID_SET, "Gaia source_id mismatch or precision loss"
assert gaia["source_id"].is_unique, "Gaia source_id rows are not unique"
missing_gaia_columns = [c for c in GAIA_FIELDS if c not in gaia.columns]
assert not missing_gaia_columns, f"Missing Gaia columns: {missing_gaia_columns}"

correlation_cols = [c for c in gaia.columns if c.endswith("_corr")]
for col in correlation_cols:
    vals = pd.to_numeric(gaia[col], errors="coerce").dropna()
    assert ((vals >= -1) & (vals <= 1)).all(), f"Correlation outside [-1,1]: {col}"

print("Gaia validation passed")
print("Correlation columns:", correlation_cols)

Gaia validation passed
Correlation columns: ['ra_dec_corr', 'ra_parallax_corr', 'ra_pmra_corr', 'ra_pmdec_corr', 'dec_parallax_corr', 'dec_pmra_corr', 'dec_pmdec_corr', 'parallax_pmra_corr', 'parallax_pmdec_corr', 'pmra_pmdec_corr']


In [6]:
# Field-level inventory: local pre-query availability plus retrieved Gaia DR3 values.
field_records = []
for _, row in local_best.iterrows():
    field_records.append({
        "audit_stage": "local_pre_query",
        "source_id": row["source_id"],
        "field": row["field"],
        "source_table": row["source_table"],
        "availability": row["availability"],
        "value": row["value"],
        "unit": row["unit"],
        "provenance": row["provenance"],
        "missingness": row["missingness"],
        "intended_mc_role": row["intended_mc_role"],
    })

for _, grow in gaia.iterrows():
    sid = grow["source_id"]
    for field in GAIA_FIELDS:
        value = grow[field] if field in gaia.columns else pd.NA
        availability = "available" if pd.notna(value) and str(value) != "" else "column_present_value_missing"
        field_records.append({
            "audit_stage": "gaia_dr3_retrieved",
            "source_id": sid,
            "field": field,
            "source_table": "gaiadr3.gaia_source",
            "availability": availability,
            "value": "" if pd.isna(value) else str(value),
            "unit": FIELD_UNITS.get(field, "Gaia DR3 catalog-defined"),
            "provenance": "Gaia TAP query saved to data/raw/project_vi_priority_a_gaia_dr3_astrometry.csv",
            "missingness": "not missing" if availability == "available" else "value missing",
            "intended_mc_role": intended_role(field),
        })

field_inventory = pd.DataFrame(field_records)
field_inventory.to_csv(FIELD_INVENTORY_OUT, index=False)
print("Saved", FIELD_INVENTORY_OUT, field_inventory.shape)
field_inventory.head(20)

Saved /Users/liors/Documents/research/gaia-lamost-galactic-archaeology/data/processed/project_vi_priority_a_field_inventory.csv (123, 10)


,audit_stage,source_id,field,source_table,availability,value,unit,provenance,missingness,intended_mc_role
0,local_pre_query,3089534353001157632,astrometric_params_solved,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,dimensionless,local repository file,column missing,quality/completeness audit
1,local_pre_query,3089534353001157632,dec,data/processed/project_ii_angular_momentum_can...,column_present_value_missing,,deg,local repository file,value missing,coordinate central value for orbit pipeline
2,local_pre_query,3089534353001157632,dec_error,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,mas,local repository file,column missing,coordinate uncertainty audit; not used until u...
3,local_pre_query,3089534353001157632,dec_parallax_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,retrieved for full astrometric audit; not used...
4,local_pre_query,3089534353001157632,dec_pmdec_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,retrieved for full astrometric audit; not used...
5,local_pre_query,3089534353001157632,dec_pmra_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,retrieved for full astrometric audit; not used...
6,local_pre_query,3089534353001157632,duplicated_source,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,boolean,local repository file,column missing,quality/completeness audit
7,local_pre_query,3089534353001157632,parallax,data/processed/gaia_lamost_candidate_diagnosti...,available,0.4587656346030645,mas,local repository file,not missing,central value for later correlated astrometric MC
8,local_pre_query,3089534353001157632,parallax_error,data/processed/project_vi_mc_prototype_candida...,available,0.018229530984877475,mas,local repository file,not missing,standard uncertainty for later parallax/pm cov...
9,local_pre_query,3089534353001157632,parallax_pmdec_corr,data/processed/gaia_lamost_candidate_diagnosti...,missing_column,,correlation coefficient,local repository file,column missing,correlation coefficient for 3D parallax/pm cov...


In [7]:
# Audit LAMOST RV central values and any measured RV uncertainty or quality fields available locally.
rv_candidates = local_audit[
    local_audit["source_id"].isin(TARGET_ID_SET)
    & local_audit["field"].str.lower().isin(["rv", "rv_err", "radial_velocity_error", "snrg", "snru", "snrr", "snri", "snrz"])
].copy()
rv_available = rv_candidates[rv_candidates["availability"] == "available"].sort_values(["source_id", "field", "source_table"])
rv_available

,source_id,field,source_table,availability,value,unit,provenance,missingness,intended_mc_role
32,3089534353001157632,rv,data/processed/gaia_lamost_candidate_diagnosti...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
109,3089534353001157632,rv,data/processed/gaia_lamost_candidate_summary_t...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
181,3089534353001157632,rv,data/processed/gaia_lamost_larger_chemo_kinema...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
253,3089534353001157632,rv,data/processed/gaia_lamost_larger_chemo_kinema...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
361,3089534353001157632,rv,data/processed/gaia_lamost_larger_crossmatched...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
397,3089534353001157632,rv,data/processed/gaia_lamost_larger_velocity_fea...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
597,3089534353001157632,rv,data/processed/project_ii_angular_momentum_can...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
685,3089534353001157632,rv,data/processed/project_ii_distance_recovered_c...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
781,3089534353001157632,rv,data/processed/project_ii_galpy_orbit_candidat...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit
877,3089534353001157632,rv,data/processed/project_ii_orbit_angular_moment...,available,-43.26,km/s,local repository file,not missing,radial-velocity central value audit


In [8]:
# Choose RV central values conservatively. Gaia RV is audited, but the existing orbit chain uses LAMOST rv.
velocity = pd.read_csv(DATA_PROCESSED / "gaia_lamost_larger_velocity_features.csv", dtype={"source_id": "string"})
velocity["source_id"] = source_key(velocity["source_id"])
vel_targets = velocity[velocity["source_id"].isin(TARGET_ID_SET)][["source_id", "rv"]].copy()
vel_targets["rv"] = pd.to_numeric(vel_targets["rv"], errors="coerce")

rows=[]
for _, grow in gaia.iterrows():
    sid = grow["source_id"]
    vrow = vel_targets[vel_targets["source_id"] == sid].iloc[0]
    gaia_rv = pd.to_numeric(grow.get("radial_velocity"), errors="coerce")
    gaia_rv_err = pd.to_numeric(grow.get("radial_velocity_error"), errors="coerce")
    lamost_rv = float(vrow["rv"]) if pd.notna(vrow["rv"]) else np.nan

    rv_central = lamost_rv
    rv_source = "LAMOST rv from gaia_lamost_larger_velocity_features.csv"
    rv_unc = np.nan
    rv_unc_source = "missing_measured_rv_uncertainty"
    rv_unc_ready = False
    if pd.notna(gaia_rv_err) and pd.notna(gaia_rv):
        # Do not mix Gaia RV uncertainty with LAMOST central RV in the covariance-ready input.
        gaia_rv_note = "Gaia RV uncertainty exists only with Gaia RV central value; not mixed with LAMOST central RV"
    else:
        gaia_rv_note = "Gaia radial_velocity_error missing or Gaia radial_velocity missing"

    base = {
        "source_id": sid,
        "ra_deg": pd.to_numeric(grow["ra"], errors="coerce"),
        "ra_error_mas": pd.to_numeric(grow["ra_error"], errors="coerce"),
        "dec_deg": pd.to_numeric(grow["dec"], errors="coerce"),
        "dec_error_mas": pd.to_numeric(grow["dec_error"], errors="coerce"),
        "parallax_mas": pd.to_numeric(grow["parallax"], errors="coerce"),
        "parallax_error_mas": pd.to_numeric(grow["parallax_error"], errors="coerce"),
        "pmra_masyr": pd.to_numeric(grow["pmra"], errors="coerce"),
        "pmra_error_masyr": pd.to_numeric(grow["pmra_error"], errors="coerce"),
        "pmdec_masyr": pd.to_numeric(grow["pmdec"], errors="coerce"),
        "pmdec_error_masyr": pd.to_numeric(grow["pmdec_error"], errors="coerce"),
        "gaia_radial_velocity_kms": gaia_rv,
        "gaia_radial_velocity_error_kms": gaia_rv_err,
        "rv_central_kms": rv_central,
        "rv_central_source": rv_source,
        "rv_uncertainty_kms": rv_unc,
        "rv_uncertainty_source": rv_unc_source,
        "rv_uncertainty_ready": rv_unc_ready,
        "rv_uncertainty_note": gaia_rv_note,
        "ruwe": pd.to_numeric(grow["ruwe"], errors="coerce"),
        "astrometric_params_solved": grow["astrometric_params_solved"],
        "duplicated_source": grow["duplicated_source"],
        "visibility_periods_used": pd.to_numeric(grow["visibility_periods_used"], errors="coerce"),
        "gaia_source_table": "gaiadr3.gaia_source",
        "raw_query_output": str(RAW_OUT.relative_to(ROOT)),
        "coordinate_uncertainty_note": "ra_error and dec_error are in mas; not included in first 3D parallax/pm covariance matrix",
        "covariance_variable_order": "parallax_mas, pmra_masyr, pmdec_masyr",
        "covariance_units": "mas^2, mas^2/yr, mas^2/yr^2 cross-terms according to variable pair",
    }
    for col in correlation_cols:
        base[col] = pd.to_numeric(grow[col], errors="coerce")
    rows.append(base)

cov_inputs = pd.DataFrame(rows)
needed = ["parallax_mas", "parallax_error_mas", "pmra_masyr", "pmra_error_masyr", "pmdec_masyr", "pmdec_error_masyr", "parallax_pmra_corr", "parallax_pmdec_corr", "pmra_pmdec_corr"]
cov_inputs["covariance_ready"] = cov_inputs[needed].notna().all(axis=1)
cov_inputs["rv_central_ready"] = cov_inputs["rv_central_kms"].notna()
cov_inputs.to_csv(PROCESSED_OUT, index=False)
print("Saved", PROCESSED_OUT, cov_inputs.shape)
cov_inputs

Saved /Users/liors/Documents/research/gaia-lamost-galactic-archaeology/data/processed/project_vi_priority_a_covariance_inputs.csv (2, 40)


,source_id,ra_deg,ra_error_mas,dec_deg,dec_error_mas,parallax_mas,parallax_error_mas,pmra_masyr,pmra_error_masyr,pmdec_masyr,...,ra_pmra_corr,ra_pmdec_corr,dec_parallax_corr,dec_pmra_corr,dec_pmdec_corr,parallax_pmra_corr,parallax_pmdec_corr,pmra_pmdec_corr,covariance_ready,rv_central_ready
0,3089534353001157632,124.176994,0.015250,0.737449,0.011500,0.458766,0.018230,2.906364,0.018547,-36.283028,...,0.063142,0.255463,-0.231793,0.178300,0.010915,0.231354,-0.248190,-0.106592,True,True
1,3089847099636770560,122.508355,0.020588,1.280094,0.017173,0.637151,0.023816,-42.072760,0.025911,-36.356421,...,-0.027511,0.339938,-0.191175,0.289752,-0.121074,0.173574,-0.119495,-0.219415,True,True


In [9]:
def covariance_matrix(row):
    sigmas = np.array([row["parallax_error_mas"], row["pmra_error_masyr"], row["pmdec_error_masyr"]], dtype=float)
    corr = np.array([
        [1.0, row["parallax_pmra_corr"], row["parallax_pmdec_corr"]],
        [row["parallax_pmra_corr"], 1.0, row["pmra_pmdec_corr"]],
        [row["parallax_pmdec_corr"], row["pmra_pmdec_corr"], 1.0],
    ], dtype=float)
    cov = corr * np.outer(sigmas, sigmas)
    return sigmas, corr, cov

check_rows=[]
for _, row in cov_inputs.iterrows():
    sigmas, corr, cov = covariance_matrix(row)
    eig = np.linalg.eigvalsh(cov)
    symmetric = bool(np.allclose(cov, cov.T, rtol=0, atol=1e-14))
    diag_ok = bool(np.allclose(np.diag(cov), sigmas**2, rtol=1e-12, atol=1e-16))
    corr_bounds_ok = bool(np.all((corr >= -1) & (corr <= 1)))
    psd = bool(eig.min() >= -1e-14)
    try:
        np.linalg.cholesky(cov)
        cholesky_ok = True
        cholesky_note = "ok"
    except np.linalg.LinAlgError as exc:
        cholesky_ok = False
        cholesky_note = str(exc)
    check_rows.append({
        "source_id": row["source_id"],
        "variable_order": "parallax_mas, pmra_masyr, pmdec_masyr",
        "sigma_parallax_mas": sigmas[0],
        "sigma_pmra_masyr": sigmas[1],
        "sigma_pmdec_masyr": sigmas[2],
        "cov_00": cov[0,0], "cov_01": cov[0,1], "cov_02": cov[0,2],
        "cov_10": cov[1,0], "cov_11": cov[1,1], "cov_12": cov[1,2],
        "cov_20": cov[2,0], "cov_21": cov[2,1], "cov_22": cov[2,2],
        "symmetric": symmetric,
        "diagonal_equals_sigma_squared": diag_ok,
        "correlation_bounds_ok": corr_bounds_ok,
        "eigenvalues": ";".join(f"{x:.16g}" for x in eig),
        "min_eigenvalue": float(eig.min()),
        "positive_semidefinite": psd,
        "cholesky_ok": cholesky_ok,
        "cholesky_note": cholesky_note,
        "numerical_fix_applied": False,
        "numerical_fix_note": "none",
    })

cov_checks = pd.DataFrame(check_rows)
# Merge compact check columns back into processed output for one-file auditability.
cov_inputs = cov_inputs.merge(cov_checks, on="source_id", how="left", validate="one_to_one")
cov_inputs.to_csv(PROCESSED_OUT, index=False)
print("Updated", PROCESSED_OUT, cov_inputs.shape)
cov_checks

Updated /Users/liors/Documents/research/gaia-lamost-galactic-archaeology/data/processed/project_vi_priority_a_covariance_inputs.csv (2, 63)


,source_id,variable_order,sigma_parallax_mas,sigma_pmra_masyr,sigma_pmdec_masyr,cov_00,cov_01,cov_02,cov_10,cov_11,...,symmetric,diagonal_equals_sigma_squared,correlation_bounds_ok,eigenvalues,min_eigenvalue,positive_semidefinite,cholesky_ok,cholesky_note,numerical_fix_applied,numerical_fix_note
0,3089534353001157632,"parallax_mas, pmra_masyr, pmdec_masyr",0.018230,0.018547,0.014079,0.000332,0.000078,-0.000064,0.000078,0.000344,...,True,True,True,0.0001727160709221058;0.0002676848295397281;0....,0.000173,True,True,ok,False,none
1,3089847099636770560,"parallax_mas, pmra_masyr, pmdec_masyr",0.023816,0.025911,0.021411,0.000567,0.000107,-0.000061,0.000107,0.000671,...,True,True,True,0.0004021086945343185;0.0005015638425774226;0....,0.000402,True,True,ok,False,none


In [10]:
# Final execution checks. No random sampling is run in this notebook.
assert len(cov_inputs) == 2
assert set(cov_inputs["source_id"]) == TARGET_ID_SET
assert cov_inputs["source_id"].is_unique
assert cov_inputs["covariance_ready"].all(), "At least one target is not covariance-ready for parallax/pm 3D covariance"
assert cov_inputs["symmetric"].all()
assert cov_inputs["diagonal_equals_sigma_squared"].all()
assert cov_inputs["correlation_bounds_ok"].all()
assert cov_inputs["positive_semidefinite"].all()
assert cov_inputs["cholesky_ok"].all()
assert not cov_inputs["rv_uncertainty_ready"].any(), "Unexpected measured RV uncertainty readiness; review RV source policy"
print("All covariance retrieval checks passed. No Monte Carlo sampling was run.")

All covariance retrieval checks passed. No Monte Carlo sampling was run.


In [11]:
def md_value(value, digits=6):
    if pd.isna(value):
        return "missing"
    if isinstance(value, (float, np.floating)):
        return f"{value:.{digits}g}"
    return str(value)

lines = [
    "# Project VI - Priority-A Full Astrometric Covariance Retrieval",
    "",
    "## Scope",
    "",
    "This report documents Gaia DR3 astrometric uncertainty and correlation retrieval for exactly two Project VI `validation_priority_A` candidates. It is an input audit for later correlated astrometric Monte Carlo work. No random sampling or new classification robustness test is performed in this stage.",
    "",
    "Targets:",
    "",
]
for sid in TARGET_IDS:
    lines.append(f"- `{sid}`")

lines += [
    "",
    "## Gaia DR3 Retrieval",
    "",
    "Source table: `gaiadr3.gaia_source` via Gaia TAP. The notebook first follows the project Gaia TAP pattern, and because `astroquery` is unavailable in the execution kernel, it uses a reproducible HTTPS TAP `/sync` fallback with the same ADQL query.",
    "",
    f"Raw query output: `{RAW_OUT.relative_to(ROOT)}`",
    "",
    "The query returned exactly two unique source IDs with string-safe handling. Gaia correlation coefficients are stored as correlations, not covariances.",
    "",
    "| source_id | ra deg | dec deg | parallax mas | parallax_error mas | pmra mas/yr | pmra_error mas/yr | pmdec mas/yr | pmdec_error mas/yr | ruwe | visibility_periods_used |",
    "|:--|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|",
]
for _, row in cov_inputs.iterrows():
    lines.append(f"| {row.source_id} | {row.ra_deg:.9f} | {row.dec_deg:.9f} | {row.parallax_mas:.9f} | {row.parallax_error_mas:.9f} | {row.pmra_masyr:.9f} | {row.pmra_error_masyr:.9f} | {row.pmdec_masyr:.9f} | {row.pmdec_error_masyr:.9f} | {row.ruwe:.4f} | {int(row.visibility_periods_used)} |")

lines += [
    "",
    "## Correlation Coefficients Used for First Covariance Matrix",
    "",
    "The first correlated astrometric MC stage is expected to propagate `parallax`, `pmra`, and `pmdec`. RA/Dec uncertainties are retrieved but not included yet because Gaia RA/Dec errors are in mas while the current coordinate chain consumes RA/Dec in degrees; adding them requires an explicit unit/frame treatment.",
    "",
    "| source_id | parallax_pmra_corr | parallax_pmdec_corr | pmra_pmdec_corr |",
    "|:--|--:|--:|--:|",
]
for _, row in cov_inputs.iterrows():
    lines.append(f"| {row.source_id} | {row.parallax_pmra_corr:.6f} | {row.parallax_pmdec_corr:.6f} | {row.pmra_pmdec_corr:.6f} |")

lines += [
    "",
    "## RV Audit",
    "",
    "The existing Project II orbit chain uses LAMOST radial-velocity central values from `data/processed/gaia_lamost_larger_velocity_features.csv`. The local larger LAMOST products for these two targets do not provide a measured `rv_err` or radial-velocity quality/SNR field. Gaia DR3 provides a `radial_velocity` and `radial_velocity_error` for one target, while the other has no Gaia RV error. This stage does not mix a Gaia RV uncertainty with the LAMOST RV central value, so no measured RV uncertainty is ready for joint propagation with the current orbit-chain convention.",
    "",
    "| source_id | rv_central_kms | rv_source | Gaia radial_velocity | Gaia radial_velocity_error | rv_uncertainty_ready |",
    "|:--|--:|:--|--:|--:|:--|",
]
for _, row in cov_inputs.iterrows():
    lines.append(f"| {row.source_id} | {row.rv_central_kms:.2f} | {row.rv_central_source} | {md_value(row.gaia_radial_velocity_kms)} | {md_value(row.gaia_radial_velocity_error_kms)} | {row.rv_uncertainty_ready} |")

lines += [
    "",
    "## Covariance Matrix Definition",
    "",
    "Variable order:",
    "",
    "```text",
    "parallax_mas, pmra_masyr, pmdec_masyr",
    "```",
    "",
    "For variables `i` and `j`:",
    "",
    "```text",
    "cov(i,j) = corr(i,j) * sigma_i * sigma_j",
    "```",
    "",
    "Correlation coefficients are dimensionless and must not be interpreted as covariance values.",
    "",
    "## Covariance Checks",
    "",
    "| source_id | covariance_ready | symmetric | diag=sigma^2 | corr bounds | PSD | Cholesky | min eigenvalue | eigenvalues |",
    "|:--|:--|:--|:--|:--|:--|:--|--:|:--|",
]
for _, row in cov_inputs.iterrows():
    lines.append(f"| {row.source_id} | {row.covariance_ready} | {row.symmetric} | {row.diagonal_equals_sigma_squared} | {row.correlation_bounds_ok} | {row.positive_semidefinite} | {row.cholesky_ok} | {row.min_eigenvalue:.6g} | {row.eigenvalues} |")

lines += [
    "",
    "No numerical covariance-matrix correction was applied to either target.",
    "",
    "## Readiness Assessment",
    "",
]
for _, row in cov_inputs.iterrows():
    lines.append(f"- `{row.source_id}`: covariance-ready for the 3D parallax/pmra/pmdec astrometric covariance matrix = `{row.covariance_ready}`; measured RV uncertainty ready = `{row.rv_uncertainty_ready}`.")

lines += [
    "",
    "Both candidates now have the Gaia DR3 standard uncertainties and correlation coefficients required for a later correlated astrometric MC over parallax, pmra, and pmdec. Neither candidate currently has a measured RV uncertainty that can be consistently paired with the LAMOST RV central value used by the existing orbit chain.",
    "",
    "This stage does not produce a new classification or robustness conclusion. It only establishes input readiness for the next Project VI correlated astrometric MC step.",
    "",
    "## Outputs",
    "",
    "- `notebooks/32_project_vi_priority_a_covariance_retrieval.ipynb`",
    "- `data/raw/project_vi_priority_a_gaia_dr3_astrometry.csv`",
    "- `data/processed/project_vi_priority_a_covariance_inputs.csv`",
    "- `data/processed/project_vi_priority_a_field_inventory.csv`",
    "- `report/project_vi_priority_a_covariance_retrieval.md`",
]
REPORT_OUT.write_text("\n".join(lines) + "\n")
print("Saved", REPORT_OUT)

Saved /Users/liors/Documents/research/gaia-lamost-galactic-archaeology/report/project_vi_priority_a_covariance_retrieval.md
